In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tvDatafeed import TvDatafeed, Interval

tv = TvDatafeed()

spx = tv.get_hist(
    symbol = 'SPX',
    exchange = 'TVC',
    n_bars = 10000,
    interval = Interval.in_daily
)
spx.index = pd.to_datetime(
    spx.index.strftime('%Y-%m-%d'),
)

In [7]:
macro_features = pd.read_parquet('../data/macro_original.parquet')

In [13]:
macro_features = pd.concat(
    [
        macro_features,
        spx['close'].rename('SPX'),
    ], axis = 1
).loc['2003':].ffill()

In [44]:
macro_features = macro_features.resample('ME').last()

### GPT API

pd.DataFrame형태의 데이터를 이용한 조건부 전망 생성

In [5]:
import openai

with open('../config/api.key') as file :
    lines = file.readlines()
    api_key = lines[0].strip()

openai.api_key = api_key

In [9]:
import json

def month_end_range(df: pd.DataFrame, start="2015-01-01"):
    idx = pd.to_datetime(df.index)
    df = df.copy()
    df.index = idx
    df = df.loc[start:]
    me = pd.date_range(df.index.min(), df.index.max(), freq="M")
    me = [d for d in me if d in df.index or (df.index[df.index <= d].max() is not pd.NaT)]
    return pd.DatetimeIndex(me)

def _safe_last_leq_index(idx: pd.DatetimeIndex, t: pd.Timestamp):
    sub = idx[idx <= t]
    return None if len(sub) == 0 else sub.max()

def _zscore(x: pd.Series, win: int):
    mu = x.rolling(win).mean()
    sd = x.rolling(win).std(ddof=0)
    return (x - mu) / sd

def build_feature_template(
    macro_features: pd.DataFrame,
    asof: pd.Timestamp,
    feature: str,
    horizons=(21,),                # 1개월(거래일) 기준
    lookbacks=(21, 63, 126, 252),   # 1/3/6/12개월
) -> dict:
    df = macro_features.copy()
    df.index = pd.to_datetime(df.index)
    idx = df.index

    t0 = _safe_last_leq_index(idx, asof)
    if t0 is None or feature not in df.columns:
        return {}

    s = df[feature].dropna()
    s = s.loc[:t0]
    if len(s) < max(lookbacks) + 5:
        # 데이터 부족하면 가능한 범위로만 생성
        lookbacks = tuple(lb for lb in lookbacks if lb < len(s))

    last = float(s.iloc[-1])
    out = {
        "feature": feature,
        "asof": str(pd.Timestamp(t0).date()),
        "last_value": last,
        "changes": {},
        "vol": {},
        "zscore": {},
        "recent_path": {
            "last_5": [float(v) for v in s.tail(5).values],
            "last_21": [float(v) for v in s.tail(21).values] if len(s) >= 21 else [float(v) for v in s.values],
        },
    }

    # 변화량(레벨/퍼센트 둘 다) + 변동성 + z-score
    for lb in lookbacks:
        if len(s) <= lb:
            continue
        delta = float(s.iloc[-1] - s.iloc[-1 - lb])
        pct = float(s.pct_change(lb).iloc[-1]) if np.isfinite(s.pct_change(lb).iloc[-1]) else None
        out["changes"][f"delta_{lb}d"] = delta
        out["changes"][f"pct_{lb}d"] = pct

        # 롤링 변동성(일간 변화 기준)
        d1 = s.diff()
        vol = float(d1.rolling(lb).std(ddof=0).iloc[-1]) if len(d1) >= lb else float(d1.std(ddof=0))
        out["vol"][f"std_diff1_{lb}d"] = vol

        z = _zscore(s, lb).iloc[-1]
        out["zscore"][f"z_{lb}d"] = float(z) if np.isfinite(z) else None

    # 목표: 1개월 ahead 변화량을 예측하도록 유도할 입력
    # horizon별로 “최근 horizon 수익/변화”를 추가
    for h in horizons:
        if len(s) > h:
            out["changes"][f"delta_{h}d"] = float(s.iloc[-1] - s.iloc[-1 - h])
            ph = s.pct_change(h).iloc[-1]
            out["changes"][f"pct_{h}d"] = float(ph) if np.isfinite(ph) else None

    return out

In [53]:
import logging
import time

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

client = openai.OpenAI(api_key=openai.api_key)

def build_macro_prompt(var_name: str, history: pd.Series, spx_ret_3m: float) -> str:
    hist_tail = history.tail(12).to_string()

    prompt = f"""
You are a professional macroeconomic forecaster.

Target variable: {var_name}

Recent 12 monthly observations:
{hist_tail}

S&P 500 3-month return: {spx_ret_3m:.4f}

Instructions:
- Produce a 1-month-ahead forecast for the target variable.
- The forecast must be a continuous numeric value (not discrete, not categorical).
- Provide a confidence score between 0 and 1.
- Provide a short rationale focusing only on macro drivers.
- Do NOT reveal chain-of-thought or internal reasoning steps.
- Do NOT include any text outside JSON.
- Output must strictly follow the JSON schema below.

Output JSON schema:
{{
  "forecast": float,
  "confidence": float,
  "rationale": "short text"
}}
"""
    return prompt.strip()

def gpt_forecast_from_template(prompt: str, model: str = "gpt-5-mini"):
    resp = client.responses.create(
        model=model,
        input=prompt,
        max_output_tokens=800,   # 충분히 크게
        reasoning={"effort": "low"},  # reasoning 최소화
        text={"format": {"type": "text"}},  # 최종 텍스트 강제
        store=False
    )

    content = None

    # Responses API는 output_text가 가장 안정적
    if hasattr(resp, "output_text") and resp.output_text:
        content = resp.output_text
    else:
        # fallback
        try:
            content = resp.output[0].content[0].text
        except Exception:
            pass

    if content is None:
        raise RuntimeError(f"Empty response from model={model}: {resp}")

    parsed = json.loads(content)
    forecast = float(parsed["forecast"])
    confidence = float(parsed["confidence"])
    rationale = parsed["rationale"]

    return forecast, confidence, rationale

def generate_monthly_macro_forecasts(
    macro_features: pd.DataFrame,
    spx: pd.Series,
    forecast_start="2010-01-31",
    history_start=None,
    model="gpt-5-mini",
    sleep_sec=0.2,
    max_retries=2,
):
    data = macro_features.copy()
    if history_start is not None:
        data = data.loc[history_start:].copy()

    spx_ret_3m = spx.pct_change(63)
    month_end_idx = data.resample("ME").last().index

    # 실제 호출 대상 월만 카운트
    forecast_months = [dt for dt in month_end_idx if pd.to_datetime(dt) >= pd.to_datetime(forecast_start)]
    total_calls = len(forecast_months) * len(data.columns)

    logging.info(f"[INIT] forecast_start={forecast_start}, history_start={history_start}")
    logging.info(f"[INIT] Months to forecast: {len(forecast_months)}")
    logging.info(f"[INIT] Total API calls (planned): {total_calls}")

    records = []
    done_calls = 0

    for m_i, dt in enumerate(month_end_idx, 1):
        if pd.to_datetime(dt) < pd.to_datetime(forecast_start):
            continue

        spx_val = spx_ret_3m.loc[dt] if dt in spx_ret_3m.index else 0.0

        logging.info(
            f"[MONTH {m_i}/{len(month_end_idx)}] "
            f"date={dt.date()} | done={done_calls}/{total_calls} | remaining={total_calls - done_calls}"
        )

        for v_i, var in enumerate(data.columns, 1):
            hist = data[var].loc[:dt]
            if len(hist) < 12:
                done_calls += 1
                logging.warning(f"[SKIP] {dt.date()} | {var} | insufficient history")
                continue

            logging.info(
                f"[CALL] {done_calls+1}/{total_calls} | {dt.date()} | {var} | sending request..."
            )

            prompt = build_macro_prompt(var, hist, float(spx_val))

            success = False
            for retry in range(max_retries + 1):
                try:
                    fcst, conf, rationale = gpt_forecast_from_template(prompt, model=model)

                    records.append({
                        "date": dt,
                        "variable": var,
                        "forecast_1m": float(fcst),
                        "confidence": float(conf),
                        "rationale": rationale,
                    })

                    logging.info(
                        f"[OK] {done_calls+1}/{total_calls} | {dt.date()} | {var} "
                        f"| fcst={float(fcst):.4f}, conf={float(conf):.2f}"
                    )
                    success = True
                    break

                except Exception as e:
                    logging.error(
                        f"[FAIL] {dt.date()} | {var} | retry={retry}/{max_retries} | {repr(e)}"
                    )
                    time.sleep(1.0)

            if not success:
                records.append({
                    "date": dt,
                    "variable": var,
                    "forecast_1m": None,
                    "confidence": None,
                    "rationale": None,
                })
                logging.error(f"[DROP] {dt.date()} | {var} | all retries failed")

            done_calls += 1

            logging.info(
                f"[PROGRESS] done={done_calls}/{total_calls} | remaining={total_calls - done_calls}"
            )

            time.sleep(sleep_sec)

    df = pd.DataFrame(records)
    logging.info(f"[DONE] Total rows generated: {len(df)}")

    return df

In [55]:
import logging, sys

root = logging.getLogger()
root.handlers.clear()  # 이미 설정된 핸들러 있으면 제거

handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)
handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))

root.addHandler(handler)
root.setLevel(logging.INFO)

2026-02-15 19:07:00,067 | INFO | [TEST] logging is working


In [57]:
forecast_df = generate_monthly_macro_forecasts(
    macro_features = macro_features.drop('SPX', axis=1),
    spx = macro_features['SPX'],
    forecast_start = "2014-12-31",
    history_start = None,
    model = "gpt-5-mini",
    sleep_sec = 0.2,
    max_retries = 2,
)

2026-02-15 20:37:01,553 | INFO | [INIT] forecast_start=2014-12-31, history_start=None
2026-02-15 20:37:01,554 | INFO | [INIT] Months to forecast: 135
2026-02-15 20:37:01,554 | INFO | [INIT] Total API calls (planned): 1080
2026-02-15 20:37:01,573 | INFO | [MONTH 144/278] date=2014-12-31 | done=0/1080 | remaining=1080
2026-02-15 20:37:01,574 | INFO | [CALL] 1/1080 | 2014-12-31 | BEI | sending request...
2026-02-15 20:37:05,810 | INFO | HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2026-02-15 20:37:05,823 | INFO | [OK] 1/1080 | 2014-12-31 | BEI | fcst=1.6200, conf=0.60
2026-02-15 20:37:05,824 | INFO | [PROGRESS] done=1/1080 | remaining=1079
2026-02-15 20:37:06,030 | INFO | [CALL] 2/1080 | 2014-12-31 | VIX | sending request...
2026-02-15 20:37:09,384 | INFO | HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2026-02-15 20:37:09,388 | INFO | [OK] 2/1080 | 2014-12-31 | VIX | fcst=15.5000, conf=0.60
2026-02-15 20:37:09,389 | INFO | [PROGRESS] 

In [59]:
forecast_df.to_parquet('../data/macro_forecasts_gpt.parquet')

In [63]:
forecast_df

,date,variable,forecast_1m,confidence,rationale
0,2014-12-31,BEI,1.620,0.60,BEI has shown a steady downtrend over recent m...
1,2014-12-31,VIX,15.500,0.60,Recent VIX series shows mean-reversion after i...
2,2014-12-31,USOIL,48.600,0.35,Oil prices likely continue lower month‑to‑mont...
3,2014-12-31,TBILL,0.042,0.60,Strong equity returns and improving risk appet...
4,2014-12-31,COPPER,2.780,0.60,Expect modest near-term softness for copper fr...
...,...,...,...,...,...
1075,2026-02-28,TBILL,3.720,0.60,Recent downward trend in short-term yields has...
1076,2026-02-28,COPPER,5.950,0.60,Macro drivers point to continued modest indust...
1077,2026-02-28,DXY,96.300,0.60,Moderate dollar weakness expected over the nex...
1078,2026-02-28,HYS,-0.820,0.60,Modest improvement in risk sentiment (positive...


In [64]:
forecast_1m_df = (
    forecast_df
    .pivot(index="date", columns="variable", values="forecast_1m")
    .sort_index()
)

In [65]:
confidence_df = (
    forecast_df.pivot(
        index = 'date', columns = 'variable', values = 'confidence'
    ).sort_index()
)

In [74]:
macro_view = (forecast_1m_df / macro_features.drop('SPX', axis = 1) - 1).loc['2014-12':]

In [87]:
macro_view.loc[:,'TBILL'] = (forecast_1m_df['TBILL'] - macro_features['TBILL'])
macro_view.loc[:,'HYS'] = (forecast_1m_df['HYS'] - macro_features['HYS'])

In [132]:
confidence_df.to_parquet('../data/macro_confidence.parquet')
forecast_1m_df.to_parquet('../data/macro_forecasts.parquet')
macro_view.to_parquet('../data/macro_view.parquet')

### RAG

In [127]:
from typing import List, Dict
from sklearn.metrics.pairwise import cosine_similarity
import re

def safe_json_loads(txt: str):
    if txt is None:
        raise ValueError("Empty response")

    txt = txt.strip()

    m = re.search(r"\{.*\}", txt, re.S)
    if not m:
        raise ValueError(f"No JSON object found: {txt[:200]}")

    return json.loads(m.group(0))

def safe_json_extract(text: str):
    if text is None:
        return None
    text = text.strip()

    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return None

    try:
        return json.loads(text[start:end+1])
    except Exception:
        return None

def rag_decide(prompt: str, model="gpt-5-mini", max_retries=2):
    last_err = None

    for retry in range(max_retries + 1):
        try:
            resp = client.responses.create(
                model=model,
                input=prompt,
                max_output_tokens=120,
            )

            content = None
            if hasattr(resp, "output_text") and resp.output_text:
                content = resp.output_text
            elif resp.output and resp.output[0].content:
                content = resp.output[0].content[0].text

            parsed = safe_json_extract(content)

            if parsed is None:
                raise ValueError("Invalid JSON")

            return bool(parsed["use"]), float(parsed["adjustment"]), parsed["reason"]

        except Exception as e:
            last_err = e
            time.sleep(0.5)

    raise RuntimeError(f"RAG JSON parse failed after retries: {repr(last_err)}")

class SimpleVectorStore:
    def __init__(self, dim: int):
        self.embs = []
        self.meta = []
        self.dim = dim

    def add(self, emb: np.ndarray, meta: Dict):
        self.embs.append(emb.reshape(1, -1))
        self.meta.append(meta)

    def search(self, q: np.ndarray, k: int = 5):
        if len(self.embs) == 0:
            return []
        M = np.vstack(self.embs)
        sims = cosine_similarity(q.reshape(1, -1), M)[0]
        idx = np.argsort(-sims)[:k]
        return [(self.meta[i], sims[i]) for i in idx]

def embed_texts(texts: List[str], model="text-embedding-3-large"):
    resp = client.embeddings.create(model=model, input=texts)
    return [np.array(x.embedding, dtype=float) for x in resp.data]

def build_rag_prompt(var, date, forecast, conf, retrieved_context: List[str]) -> str:
    ctx = "\n\n".join(retrieved_context)

    prompt = f"""
You are a cautious macro portfolio manager.

You MUST return ONLY valid JSON.
Do NOT include any explanation or text outside JSON.
Do NOT use markdown.

Target variable: {var}
Date: {date}
Model forecast (1M): {forecast:.4f}
Model confidence: {conf:.2f}

Retrieved historical context:
{ctx}

Return STRICT JSON:
{{
  "use": true/false,
  "adjustment": float,
  "reason": "short text"
}}

Return JSON only.
No explanation.
No markdown.
No extra text.
The response must be a single JSON object.
"""
    return prompt.strip()

def rag_decide(prompt: str, model="gpt-5-mini"):
    resp = client.responses.create(
        model=model,
        input=prompt,
        max_output_tokens=120,
    )
    content = resp.output_text
    if content is None and resp.output:
        content = resp.output[0].content[0].text

    parsed = json.loads(content)
    return bool(parsed["use"]), float(parsed["adjustment"]), parsed["reason"]

def build_vector_store_from_history(history_logs: pd.DataFrame):
    """
    history_logs columns:
    [date, variable, forecast_1m, realized_change_1m, confidence,
     error, abs_error, sq_error, hit_sign,
     rolling_mae_12m, rolling_rmse_12m, rolling_hit_12m, n_hist, has_realized, enough_history]
    """
    texts = []
    metas = []

    for _, r in history_logs.iterrows():
        txt = (
            f"Date={r['date']}, Var={r['variable']}, "
            f"Forecast={r['forecast_1m']:.3f}, "
            f"Realized={r['realized_change_1m']:.3f}, "
            f"Error={r['error']:.3f}, "
            f"HitRate12M={r['rolling_hit_12m'] if pd.notna(r['rolling_hit_12m']) else 'NA'}, "
            f"MAE12M={r['rolling_mae_12m'] if pd.notna(r['rolling_mae_12m']) else 'NA'}"
        )
        texts.append(txt)
        metas.append({
            "text": txt,
            "variable": r["variable"],
            "date": pd.to_datetime(r["date"])
        })

    embs = embed_texts(texts)
    store = SimpleVectorStore(dim=len(embs[0]))
    for e, m in zip(embs, metas):
        store.add(e, m)
    return store

def rag_filter_forecasts(
    forecast_1m_df: pd.DataFrame,
    confidence_df: pd.DataFrame,
    vector_store: SimpleVectorStore,
    k: int = 3,
    model="gpt-5-mini",
    sleep_sec=0.1,
):
    filtered = forecast_1m_df.copy()
    decision_log = []

    for dt in forecast_1m_df.index:
        for var in forecast_1m_df.columns:
            fcst = forecast_1m_df.loc[dt, var]
            conf = confidence_df.loc[dt, var]

            if pd.isna(fcst) or pd.isna(conf):
                filtered.loc[dt, var] = np.nan
                continue

            try:
                query_text = f"Var={var}, Date={dt}, Forecast={fcst:.3f}, Confidence={conf:.2f}"
                q_emb = embed_texts([query_text])[0]

                retrieved = vector_store.search(q_emb, k=k)
                retrieved_ctx = [m["text"] for m, _ in retrieved]

                prompt = build_rag_prompt(var, str(pd.to_datetime(dt).date()), float(fcst), float(conf), retrieved_ctx)
                use, adj, reason = rag_decide(prompt, model=model)

                filtered.loc[dt, var] = float(fcst) * float(adj) if use else 0.0

            except Exception as e:
                logging.error(f"[RAG_FAIL] {dt.date()} | {var} | {repr(e)}")
                filtered.loc[dt, var] = 0.0
                use, adj, reason = False, 0.0, "parse_fail_or_empty"

            decision_log.append({
                "date": dt,
                "variable": var,
                "use": use,
                "adjustment": adj,
                "reason": reason
            })

            time.sleep(sleep_sec)

    return filtered, pd.DataFrame(decision_log)

In [125]:
def build_history_logs(
    realized_df: pd.DataFrame,
    forecast_1m_df: pd.DataFrame,
    confidence_df: pd.DataFrame | None = None,
    horizon_months: int = 1,
    min_hist_months: int = 12,
    clip_conf: tuple[float, float] = (0.0, 1.0),
):
    realized_df = realized_df.copy()
    forecast_1m_df = forecast_1m_df.copy()
    if confidence_df is not None:
        confidence_df = confidence_df.copy()

    # 인덱스를 date 컬럼으로 강제 변환
    realized_df = realized_df.reset_index().rename(columns={realized_df.index.name or "index": "date"})
    forecast_1m_df = forecast_1m_df.reset_index().rename(columns={forecast_1m_df.index.name or "index": "date"})
    if confidence_df is not None:
        confidence_df = confidence_df.reset_index().rename(columns={confidence_df.index.name or "index": "date"})

    realized_df["date"] = pd.to_datetime(realized_df["date"])
    forecast_1m_df["date"] = pd.to_datetime(forecast_1m_df["date"])
    if confidence_df is not None:
        confidence_df["date"] = pd.to_datetime(confidence_df["date"])

    vars_common = sorted(list(set(realized_df.columns) & set(forecast_1m_df.columns) - {"date"}))
    realized_df = realized_df[["date"] + vars_common]
    forecast_1m_df = forecast_1m_df[["date"] + vars_common]
    if confidence_df is not None:
        confidence_df = confidence_df[["date"] + vars_common]

    realized_df = realized_df.set_index("date")
    forecast_1m_df = forecast_1m_df.set_index("date")
    if confidence_df is not None:
        confidence_df = confidence_df.set_index("date")

    realized_change = realized_df.shift(-horizon_months) - realized_df

    fc_long = forecast_1m_df.stack().reset_index()
    fc_long.columns = ["date", "variable", "forecast_1m"]

    rl_long = realized_change.stack().reset_index()
    rl_long.columns = ["date", "variable", f"realized_change_{horizon_months}m"]

    out = fc_long.merge(rl_long, on=["date", "variable"], how="left")

    if confidence_df is not None:
        cf_long = confidence_df.stack().reset_index()
        cf_long.columns = ["date", "variable", "confidence"]
        out = out.merge(cf_long, on=["date", "variable"], how="left")
        out["confidence"] = out["confidence"].clip(*clip_conf)
    else:
        out["confidence"] = np.nan

    y = out[f"realized_change_{horizon_months}m"].astype(float)
    f = out["forecast_1m"].astype(float)

    out["error"] = y - f
    out["abs_error"] = out["error"].abs()
    out["sq_error"] = out["error"] ** 2
    out["hit_sign"] = ((np.sign(y) == np.sign(f)) & (y != 0) & (f != 0)).astype(float)

    out = out.sort_values(["variable", "date"]).reset_index(drop=True)

    def add_roll_stats(g):
        g["rolling_mae_12m"] = g["abs_error"].shift(1).rolling(min_hist_months).mean()
        g["rolling_rmse_12m"] = np.sqrt(g["sq_error"].shift(1).rolling(min_hist_months).mean())
        g["rolling_hit_12m"] = g["hit_sign"].shift(1).rolling(min_hist_months).mean()
        g["n_hist"] = g["abs_error"].shift(1).rolling(min_hist_months).count()
        return g

    out = out.groupby("variable", group_keys=False).apply(add_roll_stats)

    out["has_realized"] = out[f"realized_change_{horizon_months}m"].notna()
    out["enough_history"] = out["n_hist"].fillna(0) >= min_hist_months

    return out

In [102]:
realized_df = macro_features.drop(columns=["SPX"], errors="ignore").resample("ME").last()

In [107]:
history_logs = build_history_logs(realized_df, forecast_1m_df, confidence_df, horizon_months=1)

/var/folders/1q/pl9tj55n57s9jg28npxw61n80000gn/T/ipykernel_70445/4109836398.py:71: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  out = out.groupby("variable", group_keys=False).apply(add_roll_stats)


In [112]:
history_logs.columns

Index(['date', 'variable', 'forecast_1m', 'realized_change_1m', 'confidence',
       'error', 'abs_error', 'sq_error', 'hit_sign', 'rolling_mae_12m',
       'rolling_rmse_12m', 'rolling_hit_12m', 'n_hist', 'has_realized',
       'enough_history'],
      dtype='object')

In [115]:
vector_store = build_vector_store_from_history(history_logs)

2026-02-15 22:23:38,865 | INFO | HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


In [129]:
# filtered_forecast_df, rag_log = rag_filter_forecasts(
#     forecast_1m_df = forecast_1m_df,
#     confidence_df = confidence_df,
#     vector_store = vector_store,
#     k = 3,
#     model = "gpt-5-mini",
#     sleep_sec = 0.2,
# )